# Hybrid Fusion: Combining Transformer and Lexicon-Based Scores

This notebook combines the DistilBERT baseline outputs with snippet/lexicon scores
to create a hybrid sentiment representation. The goal is to balance transformer confidence
with rule-based interpretability.

In [1]:
# 1. Setup
import pandas as pd

# Load transformer baseline predictions
bert_df = pd.read_csv("../data/processed/distilbert_baseline.csv")

# Load snippet/lexicon scores
snippet_df = pd.read_csv("../data/processed/snippet_scores.csv")

print(f"✅ Loaded BERT predictions: {bert_df.shape}")
print(f"✅ Loaded Snippet scores: {snippet_df.shape}")

✅ Loaded BERT predictions: (14166, 5)
✅ Loaded Snippet scores: (9, 5)


In [2]:
# 2. Quick sanity checks
print("BERT columns:", bert_df.columns.tolist())
print("Snippet columns:", snippet_df.columns.tolist())

print("Companies:", snippet_df['company'].unique())
print("Years:", snippet_df['year'].unique())

BERT columns: ['company', 'year', 'sentence', 'label', 'score']
Snippet columns: ['company', 'year', 'pos', 'neg', 'score']
Companies: ['Google' 'HSBC' 'Nestle']
Years: [2022 2023 2024]


In [3]:
# 3. Aggregate BERT predictions to document-level
# (since snippet_df is at document-level, we need to match granularity)
bert_doc = (
    bert_df.groupby(["company", "year"])
    .agg({
        "score": "mean"  # average confidence score across sentences
    })
    .reset_index()
    .rename(columns={"score": "bert_mean_score"})
)

print("✅ Aggregated BERT predictions to doc-level:", bert_doc.shape)

✅ Aggregated BERT predictions to doc-level: (9, 3)


In [4]:
# 4. Merge both sources
hybrid_df = pd.merge(snippet_df, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())

✅ Hybrid DataFrame created: (9, 6)
  company  year  pos  neg  score  bert_mean_score
0  Google  2022  140  107     33         0.898745
1  Google  2023  951  654    297         0.937662
2  Google  2024  689  705    -16         0.927499
3    HSBC  2022  818  672    146         0.942709
4    HSBC  2023  829  785     44         0.938589


In [5]:
# 5. Simple fusion strategy
# Example: weighted average (70% BERT, 30% lexicon)
hybrid_df["hybrid_score"] = (
    0.7 * hybrid_df["bert_mean_score"] + 0.3 * hybrid_df["score"]
)

# Save results
out_path = "../data/processed/hybrid_scores.csv"
hybrid_df.to_csv(out_path, index=False)

print(f"✅ Saved hybrid scores to {out_path}")


✅ Saved hybrid scores to ../data/processed/hybrid_scores.csv
